# Parsing Propositional Logic

Run this notebook from top to bottom with a fresh .NET (F#) kernel.  The scanner and the parser are generated by the
F# port of `Larkita` from the grammar `logic.lark`.

In [ ]:
// Nested tuples, the representation of abstract syntax trees; loaded once per session.
#load "../Support/Nested.fsx"

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#load "../Support/Larkita/Larkita.fsx"
open Larkita
open Nested

## Grammar and syntax trees

The grammar encodes the precedences structurally:  equivalence is non-associative, since its two operands are
implications; implication associates to the right, disjunction and conjunction to the left, and negation binds most
tightly.  The string literals of an alternative form the head of the abstract syntax tree, e.g. `p ∧ q` yields
the nested tuple `("∧", ("ID", "p"), ("ID", "q"))`, while `⊤` yields `("⊤",)`.  The functions below work directly on
these nested tuples.

In [ ]:
printfn "%s" (System.IO.File.ReadAllText "logic.lark")

In [ ]:
let logicParser = withConflictWarnings LarkitaAst.Raise (fun () -> Larkita.Open("logic.lark", outFile = "logic.out"))

module Parse =
    let logic text = parse logicParser text

The output file `logic.out` describes the states of the canonical LR(1) automaton.  The grammar has no conflicts.

In [ ]:
printfn "%s" (System.IO.File.ReadAllText "logic.out")

In [ ]:
#load "../Support/Visualization.fsx"
open Visualization

let test parse text =
    let expression = parse text
    printfn "%s ⇒ %s" text (toString expression)
    expression

Equivalence is non-associative: its two operands are implications. This grammar encodes that constraint structurally, Implication associates to the right, disjunction and conjunction to the left, and negation binds most tightly.

In [ ]:
let examples =
    [ "(p ↔ q) ∨ (q ↔ p) → (¬q ↔ ¬p) ∧ (q → ⊥ ↔ p → ⊥)"
      "p ∧ ⊤"; "q ∨ ⊥"; "¬p ∧ q"; "p ∧ q → r"; "p → q → r"
      "¬(p ∨ q) ↔ ¬p ∧ ¬q"; "¬¬p ↔ p"
      "(p → q) ∧ (q → r) → p → r"; "p ∨ q ∧ r"
      "v ↔ u → p ∨ q ∧ ¬ ⊥" ]
let trees = examples |> List.map (test Parse.logic)
checkNotebook (toString (Parse.logic "p → q → r") = """("→", ("ID", "p"), ("→", ("ID", "q"), ("ID", "r")))""")
checkNotebook (toString (Parse.logic "p ∨ q ∧ r") = """("∨", ("ID", "p"), ("∧", ("ID", "q"), ("ID", "r")))""")
let nonAssociative = try Parse.logic "p ↔ q ↔ r" |> ignore; false with _ -> true
checkNotebook nonAssociative

## Evaluate a formula

A valuation maps proposition names to truth values.  The function `evaluate` matches the nested tuples of a formula.  Missing propositions and unknown tuples raise an explicit error.

In [ ]:
let rec evaluate values (formula: nested) =
    match formula with
    | Node [ Leaf (Str "⊤") ] -> true
    | Node [ Leaf (Str "⊥") ] -> false
    | Node [ Leaf (Str "ID"); Leaf (Str name) ] -> Map.find name values
    | Node [ Leaf (Str "¬"); operand ] -> not (evaluate values operand)
    | Node [ Leaf (Str operator); left; right ] ->
        let p, q = evaluate values left, evaluate values right
        match operator with
        | "∧" -> p && q
        | "∨" -> p || q
        | "→" -> not p || q
        | "↔" -> p = q
        | _ -> invalidArg "formula" operator
    | other -> invalidArg "formula" (sprintf "Not a logical formula: %s" (toString other))

let tautology = Parse.logic "¬(p ∨ q) ↔ ¬p ∧ ¬q"
for p in [false; true] do
    for q in [false; true] do
        checkNotebook (evaluate (Map.ofList ["p", p; "q", q]) tautology)

In [ ]:
Microsoft.DotNet.Interactive.Kernel.display(Microsoft.DotNet.Interactive.Kernel.HTML(trees[6] |> treeToDot |> renderDot), "text/html") |> ignore